# Basic Evolution-Travel Trap Workflow (Single-Layer Demo)

This notebook demonstrates a **minimal, fully public-API** workflow:
1. Build a tiny one-layer model with a deliberately localized trap.
2. Randomize the model with `randomize_model(...)`.
3. Run `analyze_traps(...)` to detect a trap and inspect localization metrics.
4. Remove that trap with `remove_traps(...)`.

Each step includes timing with `time.perf_counter()`.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import weightwatcher as ww

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("weightwatcher version:", ww.__version__)
print("torch version:", torch.__version__)

## 1) Create a tiny single-layer model with a highly localized trap

We start from random Gaussian weights, then inject one large entry (`trap_strength`) at a single coordinate.
That creates a strongly localized outlier direction by construction.

In [ ]:
class SingleLayerTrapNet(nn.Module):
    def __init__(self, in_dim=64, out_dim=64):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x):
        return self.fc(x)

seed = 123
torch.manual_seed(seed)
np.random.seed(seed)

model = SingleLayerTrapNet(64, 64)
with torch.no_grad():
    model.fc.weight.normal_(mean=0.0, std=0.02)
    trap_row, trap_col = 7, 11
    trap_strength = 5.0
    model.fc.weight[trap_row, trap_col] = trap_strength

original_weight = model.fc.weight.detach().cpu().clone()
print("Weight shape:", tuple(original_weight.shape))
print("Injected trap location/value:", (trap_row, trap_col), float(original_weight[trap_row, trap_col]))
print("Global max |W|:", float(original_weight.abs().max()))

## 2) Randomize once with the evolution-travel workflow entry point

In [ ]:
watcher = ww.WeightWatcher(model=model)

t0 = time.perf_counter()
randomized_model, trap_state = watcher.randomize_model(model=model, layers=[], rng=123, return_state=True, pool=False)
permuted_ids = trap_state.get("permuted_ids", {})
permuted_ids = {int(k): np.asarray(v, dtype=int) for k, v in permuted_ids.items()}
trap_state["permuted_ids"] = permuted_ids
randomized_layers = sorted(permuted_ids.keys())
assert len(randomized_layers) > 0, "randomize_model did not return any permuted layer ids"
missing = [lid for lid in randomized_layers if lid not in permuted_ids]
assert not missing, f"Missing permuted_ids for randomized layers: {missing}"
for lid in randomized_layers:
    assert lid in trap_state.get("layers", {}), f"Missing trap_state layer metadata for layer {lid}"
    assert trap_state["layers"][lid].get("permuted_ids") is not None
print("randomized_layers:", randomized_layers)
print("permuted_ids keys:", sorted(permuted_ids.keys()))
print("trap_state layers:", sorted(trap_state.get("layers", {}).keys()))
randomized_checkpoint_path = "/tmp/randomized_checkpoint.pt"
torch.save({
    "model_state_dict": randomized_model.state_dict(),
    "permuted_ids": permuted_ids,
    "randomized_layers": randomized_layers,
    "trap_state": trap_state,
}, randomized_checkpoint_path)
t_randomize = time.perf_counter() - t0

rand_weight = randomized_model.fc.weight.detach().cpu().clone()
print(f"randomize_model runtime: {t_randomize:.4f} seconds")
print("Same matrix as original?", torch.allclose(original_weight, rand_weight))
print("Frobenius norm(original-randomized):", float(torch.linalg.norm(original_weight - rand_weight)))
print("Randomized layer_ids:", randomized_layers)


## 3) Re-run randomization to confirm weight matrix changes again

In [ ]:
t0 = time.perf_counter()
randomized_model_2, trap_state_2 = watcher.randomize_model(model=model, layers=[], rng=456, return_state=True, pool=False)
t_randomize2 = time.perf_counter() - t0

rand_weight_2 = randomized_model_2.fc.weight.detach().cpu().clone()
print(f"Second randomize_model runtime: {t_randomize2:.4f} seconds")
print("First randomized == second randomized?", torch.allclose(rand_weight, rand_weight_2))
print("Frobenius norm(rand1-rand2):", float(torch.linalg.norm(rand_weight - rand_weight_2)))

## 4) Detect traps with `analyze_traps(...)` and inspect localization / burden

In [ ]:
t0 = time.perf_counter()
trap_df, trap_state = watcher.analyze_traps(
    randomized_model=randomized_model,
    layers=randomized_layers,
    trap_state=trap_state,
    permuted_ids=permuted_ids,
    return_artifacts=True,
    trap_burden=True,
    trap_burden_mode="fast",
    bulk_mode_sample=10,
    trap_burden_variant="top5",
    plot=False,
    savefig=False,
    pool=False,
)
t_analyze = time.perf_counter() - t0

print(f"analyze_traps runtime: {t_analyze:.4f} seconds")
print("Number of detected traps:", len(trap_df))

cols_to_show = [
    c for c in [
        "layer_id", "name", "trap_index", "trap_mode_index",
        "matrix_entropy", "vector_entropy", "participation_ratio", "localization_ratio",
        "trap_burden", "trap_burden_top5"
    ] if c in trap_df.columns
]

if len(trap_df) > 0:
    display(trap_df[cols_to_show].head(10))
    top_trap = trap_df.iloc[[0]].copy()
    if "trap_index" in top_trap.columns:
        top_trap["trap_index"] = top_trap["trap_index"].astype(int) + 1
else:
    top_trap = None
    print("No trap rows found. Try increasing matrix size or trap_strength.")

## 5) Remove a detected trap with `remove_traps(...)`

In [ ]:
if top_trap is None:
    print("Skipping remove_traps because no trap was detected.")
else:
    t0 = time.perf_counter()
    ablated_model = watcher.remove_traps(
        randomized_model=randomized_model,
        traps=top_trap,
        trap_state=trap_state,
        plot=False,
        pool=False,
    )
    t_remove = time.perf_counter() - t0

    before = randomized_model.fc.weight.detach().cpu()
    after = ablated_model.fc.weight.detach().cpu()

    print(f"remove_traps runtime: {t_remove:.4f} seconds")
    print("Weights changed after trap removal?", not torch.allclose(before, after))
    print("Frobenius norm(before-after):", float(torch.linalg.norm(before - after)))
